# NQ-Swap: Steering Parametric vs. Context Knowledge

Language models store factual knowledge in their weights (parametric knowledge) but can also extract
answers from a provided context passage. These two sources of knowledge can conflict — a phenomenon
known as the **knowledge conflict** problem.

This notebook benchmarks state-control steering methods on the
[NQ-Swap](https://huggingface.co/datasets/pminervini/NQ-Swap) dataset, which exposes this conflict
explicitly. Each sample pairs a factual question with:

- An **original** context passage whose answer the model likely knows from training (`org_answer`).
- A **substituted** context passage with a swapped entity whose answer is inconsistent with the
  model's parametric knowledge (`sub_answer`).

The model is always prompted with the *substituted* context. We then evaluate two steering targets:

| Target | Goal | Metric |
|---|---|---|
| **Parametric** | Ignore the passage; answer from memory | `parametric_accuracy` |
| **Context** | Trust the passage; answer what it says | `context_accuracy` |

We compare a **baseline** (no steering) against **CAA** (Contrastive Activation Addition) and
**ActAdd** (Activation Addition), both steered in each direction.

### Runtime Estimate

> **Estimated Time:** 30–60 minutes (200 evaluation samples, 5 pipelines)  
> **Device:** NVIDIA A100 / H100 GPU (40 GB+ VRAM)

Reduce `N_EVAL` or `N_STEER` below to speed up experimentation.

## Setup

In [ ]:
from dotenv import load_dotenv
import os
from huggingface_hub import login

load_dotenv()
token = "HF_TOKEN"
login(token="HF_TOKEN")

/data/arshan/hallucination/AISteer360/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path

import pandas as pd
import torch
import transformers
from datasets import load_dataset

from aisteer360.algorithms.state_control.caa.control import CAA
from aisteer360.algorithms.state_control.act_add.control import ActAdd
from aisteer360.algorithms.state_control.common.specs import ContrastivePairs
from aisteer360.evaluation.benchmark import Benchmark
from aisteer360.evaluation.use_cases.nq_swap.use_case import NQSwap
from aisteer360.evaluation.metrics.custom.nq_swap.answer_exact_match import AnswerExactMatch
from aisteer360.evaluation.utils.data_utils import flatten_profiles

transformers.logging.set_verbosity_error()

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# number of samples for steering vector estimation and evaluation
N_STEER = 500
N_EVAL  = 200

NOTEBOOK_DIR = Path.cwd() / "examples/notebooks/benchmark_nq_swap"
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

## Loading the Data

We load the `dev` split of [NQ-Swap](https://huggingface.co/datasets/pminervini/NQ-Swap), which
contains 4 746 question–passage pairs. We add a string `id` field (required by the use case) and
convert the HuggingFace sequence fields to plain Python lists.

We then partition the data:
- **steer split** (first `N_STEER` samples) — used to estimate CAA steering vectors.
- **eval split** (next `N_EVAL` samples) — held out for benchmarking.

In [3]:
ds = load_dataset("pminervini/NQ-Swap", split="dev")

records = [
    {
        "id": str(i),
        "question": row["question"],
        "org_context": row["org_context"],
        "org_answer": list(row["org_answer"]),
        "sub_context": row["sub_context"],
        "sub_answer": list(row["sub_answer"]),
    }
    for i, row in enumerate(ds)
]

steer_records = records[:N_STEER]
eval_records  = records[N_STEER : N_STEER + N_EVAL]

print(f"Steering split : {len(steer_records)} samples")
print(f"Evaluation split: {len(eval_records)} samples")
print()
print("Example record:")
ex = eval_records[0]
print(f"  question   : {ex['question']}")
print(f"  org_answer : {ex['org_answer']}")
print(f"  sub_answer : {ex['sub_answer']}")
print(f"  sub_context: {ex['sub_context'][:120]}...")

Steering split : 500 samples
Evaluation split: 200 samples

Example record:
  question   : who was tammy from basketball wives married to
  org_answer : ['Kenny Anderson']
  sub_answer : ['Barry Bonds']
  sub_context: <P> Roman was once married to NBA basketball player Barry Bonds . The marriage produced 2 children . </P>...


## Building the Use Case

The `NQSwap` use case always presents the model with the *substituted* context passage.
The `AnswerExactMatch` metric then independently checks whether the response matches
`org_answer` (parametric) or `sub_answer` (context).

In [4]:
use_case = NQSwap(
    evaluation_data=eval_records,
    evaluation_metrics=[AnswerExactMatch()],
)

## Preparing Steering Data

### CAA Contrastive Pairs

CAA (Contrastive Activation Addition) estimates a direction in activation space by contrasting
two sets of texts. Critically, **both sets include the same substituted context** — the only
difference is which answer is provided:

- **Positives** — `Passage: {sub_context} Question: … Answer: {org_answer}`: the model sees
  the misleading passage but the answer comes from parametric knowledge.
- **Negatives** — `Passage: {sub_context} Question: … Answer: {sub_answer}`: the model sees
  the same passage and the answer follows the passage content.

This pairing isolates the *knowledge-source* direction in activation space while keeping the
context constant, giving a cleaner contrastive signal than removing the passage entirely.
Applying a **positive multiplier** steers the model toward parametric answers; a
**negative multiplier** steers it toward the context.

In [5]:
positives = [
    f"Passage: {r['sub_context']}\nQuestion: {r['question']}\nAnswer: {r['org_answer'][0]}"
    for r in steer_records
]
negatives = [
    f"Passage: {r['sub_context']}\nQuestion: {r['question']}\nAnswer: {r['sub_answer'][0]}"
    for r in steer_records
]

caa_pairs = ContrastivePairs(positives=positives, negatives=negatives)

print(f"Contrastive pairs: {len(caa_pairs.positives)}")
print(f"\nPositive example (sub_context + org_answer):\n  {positives[0][:300]}...")
print(f"\nNegative example (sub_context + sub_answer):\n  {negatives[0][:300]}...")

Contrastive pairs: 500

Positive example (sub_context + org_answer):
  Passage: <P> The fourth season of Chicago Fire , an American drama television series with executive producer Dick Wolf , and producers Derek Haas , Michael Brandt , and Matt Olmstead , was ordered on February 5 , 2015 , by NBC , and premiered on October 13 , 2015 and concluded on May 17 , 2016 . The...

Negative example (sub_context + sub_answer):
  Passage: <P> The fourth season of Chicago Fire , an American drama television series with executive producer Dick Wolf , and producers Derek Haas , Michael Brandt , and Matt Olmstead , was ordered on February 5 , 2015 , by NBC , and premiered on October 13 , 2015 and concluded on May 17 , 2016 . The...


## Defining the Steering Pipelines

We define five pipelines:

| Pipeline | Method | Steering target |
|---|---|---|
| `baseline` | none | — |
| `caa_parametric` | CAA (multiplier > 0) | parametric knowledge |
| `caa_context` | CAA (multiplier < 0) | context following |
| `act_add_parametric` | ActAdd | parametric knowledge |
| `act_add_context` | ActAdd | context following |

For CAA we reuse the same contrastive pairs and flip the direction via the sign of `multiplier`.
For ActAdd we supply a prompt pair that captures the same conceptual contrast, letting the
library extract a positional steering vector at inference time.

In [6]:
CAA_MULTIPLIER = 20
ACT_ADD_MULTIPLIER = 10

steering_pipelines = {
    "baseline": [],
    "caa_parametric": [
        CAA(
            data=caa_pairs,
            multiplier=CAA_MULTIPLIER,
            token_scope="after_prompt",
        )
    ],
    "caa_context": [
        CAA(
            data=caa_pairs,
            multiplier=-CAA_MULTIPLIER,
            token_scope="after_prompt",
        )
    ],
    "act_add_parametric": [
        ActAdd(
            positive_prompt="Answer the question from your own memory, ignoring any passage.",
            negative_prompt="Answer the question using only the provided passage.",
            multiplier=ACT_ADD_MULTIPLIER,
        )
    ],
    "act_add_context": [
        ActAdd(
            positive_prompt="Answer the question using only the provided passage.",
            negative_prompt="Answer the question from your own memory, ignoring any passage.",
            multiplier=ACT_ADD_MULTIPLIER,
        )
    ],
}

## Running the Benchmark

The `Benchmark` class loads the model once, applies each pipeline in turn, and saves
incremental checkpoints so the run can be resumed if interrupted.

We use greedy decoding (`do_sample=False`) with a short generation budget (`max_new_tokens=30`)
since NQ-Swap answers are short phrases.

In [10]:
benchmark = Benchmark(
    use_case=use_case,
    base_model_name_or_path=MODEL_NAME,
    steering_pipelines=steering_pipelines,
    gen_kwargs={"max_new_tokens": 30, "do_sample": False},
    save_dir=NOTEBOOK_DIR / "profiles",
    batch_size=1,
)

profiles = benchmark.run()
benchmark.export(profiles, save_dir=NOTEBOOK_DIR / "profiles")

Running pipeline: baseline...


Loading checkpoint shards: 100%|██████████| 4/4 [00:07<00:00,  1.76s/it]
Some parameters are on the meta device because they were offloaded to the cpu.


KeyboardInterrupt: 

## Results

We extract `parametric_accuracy` and `context_accuracy` from the profiles and display a
summary table. The two metrics are complementary: a method that strongly steers toward
parametric knowledge should raise `parametric_accuracy` and lower `context_accuracy`,
and vice versa.

In [ ]:
df = flatten_profiles(
    profiles,
    metric_accessors={
        "parametric_accuracy": ("AnswerExactMatch", "parametric_accuracy"),
        "context_accuracy":    ("AnswerExactMatch", "context_accuracy"),
    },
)

summary = (
    df.groupby("pipeline")[["parametric_accuracy", "context_accuracy"]]
    .mean()
    .round(3)
    .reset_index()
)

pipeline_order = list(steering_pipelines.keys())
summary["_order"] = summary["pipeline"].apply(
    lambda p: pipeline_order.index(p) if p in pipeline_order else len(pipeline_order)
)
summary = summary.sort_values("_order").drop(columns="_order").reset_index(drop=True)

summary.style.format({
    "parametric_accuracy": "{:.1%}",
    "context_accuracy":    "{:.1%}",
}).background_gradient(subset=["parametric_accuracy"], cmap="Blues") \
 .background_gradient(subset=["context_accuracy"],    cmap="Greens")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

pipelines = summary["pipeline"].tolist()
param_acc  = summary["parametric_accuracy"].tolist()
ctx_acc    = summary["context_accuracy"].tolist()

x = np.arange(len(pipelines))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 4))
bars1 = ax.bar(x - width / 2, param_acc, width, label="parametric accuracy", color="steelblue")
bars2 = ax.bar(x + width / 2, ctx_acc,   width, label="context accuracy",    color="seagreen")

ax.set_xticks(x)
ax.set_xticklabels(pipelines, rotation=20, ha="right")
ax.set_ylabel("Exact-match accuracy")
ax.set_ylim(0, 1)
ax.set_title(f"NQ-Swap benchmark — {MODEL_NAME.split('/')[-1]}")
ax.legend()
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
fig.savefig(NOTEBOOK_DIR / "figures" / "nq_swap_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")

## Inspecting Sample Responses

Browse per-question predictions across pipelines to understand qualitative behavior.

In [ ]:
N_SHOW = 5

rows = []
# Use the first pipeline's generations as the question source
first_pipeline = next(iter(profiles))
first_gens = profiles[first_pipeline][0]["generations"]

for i in range(min(N_SHOW, len(first_gens))):
    row = {
        "question":   first_gens[i]["question"],
        "org_answer": first_gens[i]["org_answer"],
        "sub_answer": first_gens[i]["sub_answer"],
    }
    for pipeline, runs in profiles.items():
        row[pipeline] = runs[0]["generations"][i]["response"]
    rows.append(row)

pd.DataFrame(rows).T

## Takeaways

- The **baseline** typically has moderate `context_accuracy` (the model reads the passage) but
  also non-trivial `parametric_accuracy` (its prior knowledge is strong for well-known facts).
- **CAA with a positive multiplier** pushes activations toward the parametric direction,
  increasing `parametric_accuracy` at the cost of `context_accuracy`.
- **CAA with a negative multiplier** (context steering) has the opposite effect.
- **ActAdd** achieves a similar directional effect via a single prompt-pair vector, making it
  simpler to configure but potentially less precise than data-driven CAA.
- The magnitude of the effect depends on the steering strength (`multiplier`);
  too large a value can degrade fluency and overall accuracy.